In [9]:
import struct
import csv
from datetime import datetime, timedelta
from pathlib import Path

# --- Paths --- #
LOGFILE_PATH = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/training/logfile/logfile raw/01-PE-LogFile")
OUTPUT_DIR = Path("/Users/soni/Github/Digital-Detectives_Thesis/data/training/logfile/logfile parsed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "01-PE-LogFile.csv"

# --- Constants --- #
EPOCH_DT = datetime(1601, 1, 1)  # FILETIME epoch

# --- Helper: FILETIME to datetime + nanoseconds --- #
def filetime_to_ns(ft):
    if ft is None or ft == 0:
        return (None, "")
    dt = EPOCH_DT + timedelta(microseconds=ft // 10)
    total_ns = ft * 100
    base = dt.strftime('%Y-%m-%dT%H:%M:%S')
    decimal_part = f"{total_ns % 1_000_000_000:09d}"[:7]
    return (dt, f"{base}.{decimal_part}Z")


In [11]:
with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "RecordOffset",
        "RecordLength",
        "RedoData_raw",
        "UndoData_raw",
        "CreationTime_Redo_dt",
        "ModificationTime_Redo_dt",
        "AccessTime_Redo_dt",
        "EntryTime_Redo_dt",
        "CreationTime_Redo_ns",
        "ModificationTime_Redo_ns",
        "AccessTime_Redo_ns",
        "EntryTime_Redo_ns",
        "CreationTime_Undo_dt",
        "ModificationTime_Undo_dt",
        "AccessTime_Undo_dt",
        "EntryTime_Undo_dt",
        "CreationTime_Undo_ns",
        "ModificationTime_Undo_ns",
        "AccessTime_Undo_ns",
        "EntryTime_Undo_ns"
    ])

    with open(LOGFILE_PATH, "rb") as logfile:
        data = logfile.read()
        offset = 0
        while offset + 4 <= len(data):
            sig = data[offset:offset+4]
            if sig != b'LFLG':  # valid $LogFile record signature
                offset += 1
                continue

            # Record length at 0x08 (little endian DWORD)
            record_length = struct.unpack_from("<I", data, offset + 0x08)[0]
            redo_offset = struct.unpack_from("<H", data, offset + 0x10)[0]
            redo_length = struct.unpack_from("<H", data, offset + 0x12)[0]
            undo_offset = struct.unpack_from("<H", data, offset + 0x14)[0]
            undo_length = struct.unpack_from("<H", data, offset + 0x16)[0]

            redo_raw = data[offset + redo_offset : offset + redo_offset + redo_length]
            undo_raw = data[offset + undo_offset : offset + undo_offset + undo_length]

            # Extract timestamps (assuming first 4 QWORD = 8 bytes each)
            def extract_ts(raw):
                ts_list = []
                for i in range(0, min(32, len(raw)), 8):  # 4 QWORD
                    ft = struct.unpack_from("<Q", raw, i)[0]
                    ts_list.append(filetime_to_ns(ft))
                while len(ts_list) < 4:
                    ts_list.append((None,""))
                return ts_list

            redo_dt_ns = extract_ts(redo_raw)
            undo_dt_ns = extract_ts(undo_raw)

            writer.writerow([
                offset,
                record_length,
                redo_raw.hex(),
                undo_raw.hex(),
                *[x[0] for x in redo_dt_ns],
                *[x[1] for x in redo_dt_ns],
                *[x[0] for x in undo_dt_ns],
                *[x[1] for x in undo_dt_ns]
            ])

            offset += record_length

print(f"✓ $LogFile parsing complete. CSV saved at: {OUTPUT_CSV}")


✓ $LogFile parsing complete. CSV saved at: /Users/soni/Github/Digital-Detectives_Thesis/data/training/logfile/logfile parsed/01-PE-LogFile.csv
